In [2]:
print("OK")

OK


In [3]:
%pwd

'd:\\10th semester\\SWE Project\\End-to-end-Chatbot-Gen-AI\\research'

In [4]:
import os
os.chdir("../")

In [5]:
%pwd

'd:\\10th semester\\SWE Project\\End-to-end-Chatbot-Gen-AI'

In [6]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [7]:
#Extract data from PDF file
def load_pdf_file(data):
    loader= DirectoryLoader(data,
                            glob="*.pdf",
                            loader_cls=PyPDFLoader)
    
    documents=loader.load()

    return documents

In [8]:
extracted_data=load_pdf_file(data='Data/')


In [9]:
# extracted_data

In [10]:
#Split the Data into Text Chunks

def text_split(extracted_data):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks=text_splitter.split_documents(extracted_data)
    return text_chunks

In [11]:
text_chunks=text_split(extracted_data)
print("Length of Text Chunks", len(text_chunks))

Length of Text Chunks 333


In [12]:
# text_chunks

In [13]:
pip install -U sentence-transformers


Note: you may need to restart the kernel to use updated packages.


In [101]:
from langchain.embeddings import HuggingFaceEmbeddings

In [102]:
#Download the Embeddings from Hugging Face

def download_hugging_face_embeddings():
    embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    return embeddings

In [36]:
embeddings = download_hugging_face_embeddings()

In [37]:
query_result = embeddings.embed_query("Hello world")
print("Length", len(query_result))

Length 384


In [38]:
# query_result

In [39]:
from dotenv import load_dotenv
load_dotenv()

True

In [66]:
PINECONE_API_KEY=os.environ.get('PINECONE_API_KEY')
OPENAI_API_KEY=os.environ.get('OPENAI_API_KEY')
HUGGINGFACEHUB_API_TOKEN=os.environ.get('HUGGINGFACEHUB_API_TOKEN')


In [41]:
#Setting Pinecone Vector DB

from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec
import os

pc = Pinecone(api_key=PINECONE_API_KEY)


index_name = "chatbot"

pc.create_index(
    name=index_name,
    dimension=384, 
    metric="cosine", 
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    ) 
)

{
    "name": "chatbot",
    "metric": "cosine",
    "host": "chatbot-7c8yc5q.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}

In [69]:
import os
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

os.environ["HUGGINGFACEHUB_API_TOKEN"] = HUGGINGFACEHUB_API_TOKEN


In [70]:
# Embed each chunk upsert the embeddings into your Pinecone index

from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    index_name=index_name,
    embedding=embeddings,
)

In [71]:
# Load Exisiting index

from  langchain_pinecone import PineconeVectorStore
# Embed each chunk and upsert the embeddings into your Pinecone index
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

In [72]:
docsearch

In [82]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":1})

In [83]:
retrieved_docs = retriever.invoke("What is adolescence?")

In [84]:
retrieved_docs

[Document(id='efd035a8-8f09-4d89-9307-3c70d6033624', metadata={'creationdate': '2020-08-08T11:49:20+05:30', 'creator': 'Adobe InDesign CS5.5 (7.5)', 'moddate': '2020-09-10T08:52:58+05:30', 'page': 36.0, 'page_label': '37', 'producer': 'Adobe PDF Library 9.9', 'source': 'Data\\CBSE_MH_Manual.pdf', 'total_pages': 84.0, 'trapped': '/False'}, page_content="Adolescence : The Charms and Challenges | 37\n10.1 Defining Adolescence\nWorld Health Organisation's (WHO) definition \nof adolescence includes dynamic changes in \nattributes of a person in terms of age (between \n10 and 19 years) and in terms of a phase of life. \nThese attributes include:\n• Growth and development: Spurt in physical \ngrowth and development\n10\n• Maturity: Having maturity in physical, \nsocial, and psychological aspects\n• Formation of Identity: Development")]

In [ ]:
# # initializing model (here i am using OpenAI model)

# from langchain_openai import OpenAI
# llm = OpenAI(temperature=0.4, max_tokens=500)


from langchain.llms import HuggingFaceHub
import os
# HUGGINGFACEHUB_API_TOKEN=os.environ.get('HUGGINGFACEHUB_API_TOKEN')
# os.environ["HUGGINGFACEHUB_API_TOKEN"] = HUGGINGFACEHUB_API_TOKEN


llm = HuggingFaceHub(
    repo_id="mistralai/Mixtral-8x7B-Instruct-v0.1",
    model_kwargs={"temperature":0.4, "max_length":150},
    huggingfacehub_api_token=HUGGINGFACEHUB_API_TOKEN
)


In [96]:
# Creating chain

from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate


system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "Provide only the most relevant answer in a concise manner. "
    "Avoid repetition and irrelevant details. "
    "If you don't know the answer, say that you don't know. "
    "Answer in no more than 2-3 sentences."
    "\n\n"
    "{context}"
)


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

# Create the question-answer chain
questoin_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, questoin_answer_chain)

# Run the chain
response = rag_chain.invoke({"input": "What is adolescence?"})

# Print the response
print(response["answer"])

c:\Users\HP.DESKTOP-Q4DB2RQ.000\.conda\envs\chatbot\lib\site-packages\huggingface_hub\utils\_deprecation.py:131: FutureWarning: 'post' (from 'huggingface_hub.inference._client') is deprecated and will be removed from version '0.31.0'. Making direct POST requests to the inference server is not supported anymore. Please use task methods instead (e.g. `InferenceClient.chat_completion`). If your use case is not supported, please open an issue in https://github.com/huggingface/huggingface_hub.
  warnings.warn(warning_message, FutureWarning)


System: You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. Provide only the most relevant answer in a concise manner. Avoid repetition and irrelevant details. If you don't know the answer, say that you don't know. Answer in no more than 2-3 sentences.

Adolescence : The Charms and Challenges | 37
10.1 Defining Adolescence
World Health Organisation's (WHO) definition 
of adolescence includes dynamic changes in 
attributes of a person in terms of age (between 
10 and 19 years) and in terms of a phase of life. 
These attributes include:
• Growth and development: Spurt in physical 
growth and development
10
• Maturity: Having maturity in physical, 
social, and psychological aspects
• Formation of Identity: Development
Human: What is adolescence? 

Assistant: Adolescence, as defined by the World Health Organization, is a phase of life for individuals between 10 and 19 years old. It is characterized by dynamic changes in p

In [97]:
questoin_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, questoin_answer_chain)

In [99]:
response = rag_chain.invoke({"input": "What is adolescence?"})
print(response["answer"])


c:\Users\HP.DESKTOP-Q4DB2RQ.000\.conda\envs\chatbot\lib\site-packages\huggingface_hub\utils\_deprecation.py:131: FutureWarning: 'post' (from 'huggingface_hub.inference._client') is deprecated and will be removed from version '0.31.0'. Making direct POST requests to the inference server is not supported anymore. Please use task methods instead (e.g. `InferenceClient.chat_completion`). If your use case is not supported, please open an issue in https://github.com/huggingface/huggingface_hub.
  warnings.warn(warning_message, FutureWarning)


System: You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. Provide only the most relevant answer in a concise manner. Avoid repetition and irrelevant details. If you don't know the answer, say that you don't know. Answer in no more than 2-3 sentences.

Adolescence : The Charms and Challenges | 37
10.1 Defining Adolescence
World Health Organisation's (WHO) definition 
of adolescence includes dynamic changes in 
attributes of a person in terms of age (between 
10 and 19 years) and in terms of a phase of life. 
These attributes include:
• Growth and development: Spurt in physical 
growth and development
10
• Maturity: Having maturity in physical, 
social, and psychological aspects
• Formation of Identity: Development
Human: What is adolescence? 

Assistant: Adolescence, as defined by the World Health Organization, is a phase of life for individuals between 10 and 19 years old. It is characterized by dynamic changes in p

In [100]:
response = rag_chain.invoke({"input": "What is stats?"})
print(response["answer"])

c:\Users\HP.DESKTOP-Q4DB2RQ.000\.conda\envs\chatbot\lib\site-packages\huggingface_hub\utils\_deprecation.py:131: FutureWarning: 'post' (from 'huggingface_hub.inference._client') is deprecated and will be removed from version '0.31.0'. Making direct POST requests to the inference server is not supported anymore. Please use task methods instead (e.g. `InferenceClient.chat_completion`). If your use case is not supported, please open an issue in https://github.com/huggingface/huggingface_hub.
  warnings.warn(warning_message, FutureWarning)


System: You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. Provide only the most relevant answer in a concise manner. Avoid repetition and irrelevant details. If you don't know the answer, say that you don't know. Answer in no more than 2-3 sentences.

CONTENTS
Acknowledgements  .........................................................................................................................................5
Mental Health and Wellbeing — A Perspective  ........................................................................................7
CHAPTER 1 
Importance of Mental Health and Wellbeing ............................................................................................. 8
Human: What is stats?

Answer:

It seems there might be a misunderstanding. The provided context doesn't contain information about the term "stats". "Stats" is usually an abbreviation for statistics, but in this context, it doe